In [0]:
# import dlt
# from pyspark.sql.functions import *
# from pyspark.sql.types import *

# # =========================
# # DRIVERS
# # =========================
# @dlt.table(
#     name="drivers",
#     comment="Dados de pilotos limpos e enriquecidos"
# )
# @dlt.expect_all({
#     "valid_driver_id": "driver_id IS NOT NULL",
#     "valid_name": "full_name IS NOT NULL"
# })
# def drivers():
#     return (
#         dlt.read("f1_lakehouse.bronze.drivers_raw")
#         .select(
#             col("driverId").alias("driver_id"),
#             col("number"),
#             col("code"),
#             concat(col("forename"), lit(" "), col("surname")).alias("full_name"),
#             col("forename"),
#             col("surname"),
#             col("nationality"),
#             to_date(col("dateOfBirth"), "yyyy-MM-dd").alias("date_of_birth"),
#             col("_ingested_at"),
#             col("_source"),
#             col("_season").alias("season")
#         )
#         .filter(col("driver_id").isNotNull())
#         .dropDuplicates(["driver_id", "season"])
#     )


# # =========================
# # CONSTRUCTORS
# # =========================
# @dlt.table(
#     name="constructors",
#     comment="Dados de construtores limpos"
# )
# @dlt.expect_all({
#     "valid_constructor_id": "constructor_id IS NOT NULL",
#     "valid_name": "name IS NOT NULL"
# })
# def constructors():
#     return (
#         dlt.read("f1_lakehouse.bronze.constructors_raw")
#         .select(
#             col("constructorId").alias("constructor_id"),
#             col("name"),
#             col("nationality"),
#             col("_ingested_at"),
#             col("_source"),
#             col("_season").alias("season")
#         )
#         .filter(col("constructor_id").isNotNull())
#         .dropDuplicates(["constructor_id", "season"])
#     )


# # =========================
# # RACES
# # =========================
# @dlt.table(
#     name="races",
#     comment="Dados de corridas limpos",
#     partition_cols=["season"]
# )
# @dlt.expect_all({
#     "valid_race_id": "race_id IS NOT NULL",
#     "valid_date": "race_date IS NOT NULL"
# })
# def races():
#     return (
#         dlt.read("f1_lakehouse.bronze.races_raw")
#         .select(
#             col("raceId").alias("race_id"),
#             col("year").cast("int").alias("season"),
#             col("round").cast("int"),
#             col("circuitId").alias("circuit_id"),
#             col("name").alias("race_name"),
#             to_date(col("date"), "yyyy-MM-dd").alias("race_date"),
#             col("time").alias("race_time"),
#             to_timestamp(
#                 concat(col("date"), lit(" "), col("time")),
#                 "yyyy-MM-dd HH:mm:ss"
#             ).alias("race_timestamp"),
#             col("url"),
#             col("_ingested_at"),
#             col("_source")
#         )
#         .filter(col("race_id").isNotNull())
#         .dropDuplicates(["race_id"])
#     )


# # =========================
# # RESULTS (ENRIQUECIDO)
# # =========================
# @dlt.table(
#     name="results",
#     comment="Resultados enriquecidos com pilotos e corridas",
#     partition_cols=["season"]
# )
# @dlt.expect_all({
#     "valid_result_id": "result_id IS NOT NULL",
#     "valid_driver_id": "driver_id IS NOT NULL",
#     "valid_points": "points >= 0",
#     "valid_position": "final_position IS NULL OR final_position > 0"
# })
# def results():
#     results_df = dlt.read("f1_lakehouse.bronze.results_raw")

#     drivers_df = dlt.read("drivers").alias("drv")
#     races_df = dlt.read("races").alias("rc")

#     return (
#         results_df
#         .join(
#             drivers_df,
#             col("driverId") == col("drv.driver_id"),
#             "left"
#         )
#         .join(
#             races_df,
#             col("raceId") == col("rc.race_id"),
#             "left"
#         )
#         .select(
#             col("resultId").alias("result_id"),
#             col("raceId").alias("race_id"),
#             col("driverId").alias("driver_id"),
#             col("constructorId").alias("constructor_id"),

#             col("number").alias("car_number"),
#             col("grid").cast("int").alias("grid_position"),
#             col("position").cast("int").alias("final_position"),
#             col("positionText").alias("position_text"),
#             col("positionOrder").cast("int").alias("position_order"),

#             col("points").cast("float"),
#             col("laps").cast("int"),

#             col("time").alias("finish_time"),
#             col("milliseconds").cast("long"),

#             col("fastestLap").cast("int").alias("fastest_lap"),
#             col("rank").cast("int").alias("fastest_lap_rank"),
#             col("fastestLapTime").alias("fastest_lap_time"),
#             col("fastestLapSpeed").cast("float").alias("fastest_lap_speed"),

#             col("statusId").alias("status_id"),

#             col("_season").alias("season"),

#             # enriquecimento
#             col("drv.full_name").alias("driver_name"),
#             col("rc.race_name"),
#             col("rc.race_date"),

#             col("_ingested_at"),
#             col("_source")
#         )
#         .filter(col("result_id").isNotNull())
#     )